# TP 1 — Estimer le prix d'une voiture d'occasion

**Machine Learning · Régression linéaire · TP en classe, sur Colab**

## Le contexte

**Roulez Malin** (un site fictif) publie des annonces de voitures d'occasion entre particuliers.
Quand un vendeur publie son annonce, il ne sait pas quel prix demander. Le site veut lui
**suggérer un prix**, et savoir **de combien cette suggestion peut se tromper**.

## Les données

2 030 annonces **fictives**, fabriquées pour ce TP, avec les défauts d'un vrai fichier.

## Avant de commencer

Faites **Fichier › Enregistrer une copie dans Drive**, et travaillez dans cette copie : sinon,
votre travail ne sera pas sauvegardé.

## Comment travailler

- Chaque étape commence par une courte explication, puis une consigne **À VOUS** : écrivez le
  code dans la cellule vide qui suit, exécutez-la (Maj + Entrée) et lisez le résultat.
- La consigne dit ce qu'il faut obtenir, pas comment : à vous de trouver le code et les
  bibliothèques. Bloqué ? Relisez la présentation, ou demandez.
- Deux cellules sont déjà écrites (le chargement du fichier et le pipeline) : exécutez-les et
  lisez-les.
- Après certains résultats, une cellule de texte **À VOUS** demande une ou deux phrases : ce que
  vous constatez, ce que vous décidez, et pourquoi. Double-cliquez dessus pour écrire.
- Les étapes sont les dix étapes de la présentation.

Ce TP ne se rend pas. Gardez-le : il vous servira de modèle pour le projet.

## 1. Préparer les outils

**À VOUS** : Importez les bibliothèques dont vous aurez besoin : pour manipuler des tableaux de données, faire des calculs, tracer des graphiques, puis découper les données, mesurer une erreur et enregistrer un modèle. Vous pouvez compléter cette cellule au fur et à mesure du TP.

In [ ]:
# À VOUS


## 2. Charger les données

Les données sont lues directement en ligne : exécutez simplement la cellule.

In [ ]:
import pandas as pd

brutes = pd.read_csv("https://raw.githubusercontent.com/reda76/ece-ml-tp/main/annonces_voitures_occasion.csv")
print(brutes.shape)
brutes.head()

| Colonne | Ce qu'elle contient |
|---|---|
| `id_annonce` | Numéro de l'annonce (ne sert pas à prédire) |
| `marque` | Marque de la voiture |
| `annee` | Année de la première mise en circulation |
| `kilometrage` | Kilomètres au compteur |
| `carburant` | Essence, Diesel, Hybride ou Électrique |
| `boite` | Boîte de vitesses : Manuelle ou Automatique |
| `puissance_ch` | Puissance en chevaux |
| `prix_eur` | **Le prix de l'annonce : ce qu'on veut prédire** |

## 3. Inspecter (étape 1)

Avant tout, on regarde ce qui manque et ce qui semble impossible.

**À VOUS** : Pour chaque colonne, affichez combien de valeurs manquent. Puis affichez un résumé statistique des colonnes de nombres : moyenne, minimum, maximum…

In [ ]:
# À VOUS


**À VOUS** : notez trois choses étranges.

- …
- …
- …

## 4. Les doublons (étape 2)

Une annonce publiée deux fois compterait double dans l'apprentissage, et ses deux copies
pourraient tomber l'une dans l'apprentissage, l'autre dans le test. On compare les lignes
**sans le numéro d'annonce**, qui change à chaque publication.

**À VOUS** : Comptez les annonces publiées deux fois, en comparant toutes les colonnes **sauf le numéro d'annonce**. Puis créez un nouveau tableau `sans_doublons` qui n'en garde qu'un exemplaire, sans modifier `brutes`, et affichez combien d'annonces il reste.

In [ ]:
# À VOUS


**À VOUS** : pourquoi supprimer les copies plutôt que les garder ?

## 5. Le périmètre (étape 3)

Le site veut suggérer un prix pour des voitures ordinaires. Deux règles, écrites à l'avance :
un prix entre 500 € et 150 000 €, et au plus 400 000 km.

**À VOUS** : Créez un tableau `propres` qui ne garde de `sans_doublons` que les annonces dont le prix est entre 500 € et 150 000 €, **et** dont le kilométrage est d'au plus 400 000 km. Affichez combien d'annonces ont été écartées.

In [ ]:
# À VOUS


**À VOUS** : justifiez chaque règle en une phrase. Que ne saura pas faire le site ?

## 6. Explorer (étape 4)

Deux graphiques, chacun pour une question : le prix baisse-t-il avec l'âge ? La marque
compte-t-elle ?

**À VOUS** : Faites deux graphiques, chacun avec un titre : le prix selon l'année (un point par annonce), et le prix médian de chaque marque (en barres).

In [ ]:
# À VOUS


**À VOUS** : que montre chaque graphique, en une phrase ?

## 7. Séparer (étape 5)

On met 20 % des annonces **au coffre** : le modèle ne les verra jamais pendant sa construction.
Elles serviront une seule fois, à la fin, à mesurer la version retenue.

**À VOUS** : Préparez `X`, les colonnes qui décrivent la voiture (tout sauf le numéro d'annonce et le prix), et `y`, le prix. Puis mettez 20 % des annonces de côté pour le test, avec un tirage reproductible (`random_state=42`). Nommez les quatre morceaux `X_appr`, `X_test`, `y_appr` et `y_test`.

In [ ]:
# À VOUS


## 8. La référence (étape 6)

Sans aucun modèle, le site pourrait suggérer à tout le monde le même prix : le prix médian.
Son erreur est le **seuil à battre**.

**À VOUS** : Calculez le prix médian des annonces d'apprentissage. Puis calculez l'erreur moyenne qu'on ferait en suggérant ce même prix à **toutes** les annonces du test. Gardez-la dans une variable `erreur_reference` et affichez-la en euros.

In [ ]:
# À VOUS


## 9. Le pipeline (étape 7)

Le pipeline enchaîne la préparation et le modèle dans un seul objet :

| Pour | Préparation |
|---|---|
| les nombres | remplir les valeurs manquantes par la médiane, puis normaliser |
| les catégories | une colonne 0/1 par valeur (encodage) |
| tout | la régression linéaire |

Tout ce que la préparation apprend (médianes, échelles, liste des marques) est appris sur les
données d'apprentissage, puis figé.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression


def fabriquer_pipeline(nombres, categories=()):
    preparations = [("nombres", Pipeline([("manquantes", SimpleImputer(strategy="median")),
                                          ("normalisation", StandardScaler())]), list(nombres))]
    if categories:
        preparations.append(("categories", OneHotEncoder(handle_unknown="ignore"), list(categories)))
    return Pipeline([("preparation", ColumnTransformer(preparations)),
                     ("modele", LinearRegression())])

## 10. Les versions (étape 8)

On part simple et on ajoute **une chose à la fois**. Chaque version est mesurée par
**validation croisée** sur les données d'apprentissage : 5 découpages, 5 erreurs. Leur moyenne
est l'erreur de la version ; leur variation dit ce que le hasard du découpage peut faire bouger.
Le coffre reste fermé.

**À VOUS** : Les quatre versions sont données. Pour chacune, mesurez l'erreur moyenne par validation croisée en 5 parts, sur les seules données d'apprentissage, avec le pipeline de l'étape 9. Rassemblez, dans un tableau, l'erreur moyenne et la variation (l'écart-type) de chaque version.

In [ ]:
VERSIONS = {
    "V1": (["annee"], []),
    "V2": (["annee", "kilometrage"], []),
    "V3": (["annee", "kilometrage", "puissance_ch"], ["carburant", "boite"]),
    "V4": (["annee", "kilometrage", "puissance_ch"], ["carburant", "boite", "marque"]),
}
# À VOUS : pour chaque version, la validation croisée, puis un tableau des résultats


**À VOUS** : quel gain apporte chaque ajout ? Est-il plus grand que la variation ? Quelle version retenez-vous ?

## 11. Mesurer la version retenue (fin de l'étape 8)

On ouvre le coffre, **une seule fois**, pour mesurer la version retenue sur des annonces
qu'elle n'a jamais vues.

**À VOUS** : Entraînez le pipeline de la version que vous retenez sur toutes les données d'apprentissage, et appelez-le `modele`. Puis mesurez son erreur sur le test, **une seule fois**, et comparez-la à `erreur_reference`.

In [ ]:
RETENUE = "V4"   # la version que vous retenez
nombres, categories = VERSIONS[RETENUE]
colonnes = nombres + categories
# À VOUS : entraîner « modele », puis mesurer son erreur sur le test


## 12. Interpréter (étape 9)

Un modèle se lit. On prend une voiture, on change **une seule chose**, et on regarde comment
le prix suggéré bouge.

**À VOUS** : Voici une voiture. Affichez le prix que `modele` lui suggère. Puis changez **une seule chose** à la fois (l'année à 2017, le kilométrage à 110 000 km, la boîte en automatique) et affichez à chaque fois l'écart de prix.

In [ ]:
voiture = pd.DataFrame([{"annee": 2018, "kilometrage": 90_000, "puissance_ch": 110,
                         "carburant": "Essence", "boite": "Manuelle", "marque": "Peugeot"}])
# À VOUS : le prix suggéré, puis l'écart quand on change une seule chose


**À VOUS** : Affichez les 5 annonces du test où le modèle se trompe le plus, avec pour chacune le vrai prix, le prix suggéré et l'écart entre les deux.

In [ ]:
# À VOUS


**À VOUS** : que vaut un an de plus, en euros ? Qu'ont en commun les plus grosses erreurs ?

## 13. Livrer (étape 10)

Le site ne refait pas tout ce notebook à chaque annonce : on enregistre le pipeline dans un
fichier, et le site le recharge. Le pipeline prépare tout seul une annonce brute, même avec une
valeur manquante.

**À VOUS** : Enregistrez `modele` dans un fichier, rechargez-le comme le ferait le site, puis suggérez un prix pour l'annonce ci-dessous, dont la puissance est inconnue.

In [ ]:
annonce = pd.DataFrame([{"annee": 2020, "kilometrage": 60_000, "puissance_ch": float("nan"),
                         "carburant": "Hybride", "boite": "Automatique", "marque": "Toyota"}])
# À VOUS : enregistrer, recharger, puis suggérer un prix pour cette annonce


## 14. Conclusion pour Roulez Malin

**À VOUS** : en quatre ou cinq phrases, sans jargon : de combien se trompe la suggestion,
comparée à la référence ? Le site peut-il l'afficher, et comment la présenter ? Quelles sont ses
limites ?